# Live pipeline: S3 raw message log -> live_delta

Flattens the Kafka raw log that `live/consumer/kafka_to_s3.py` lands in `s3://<bucket>/live/raw/date=YYYY-MM-DD/`
into the `live_delta` event tables the dashboard's Live tab reads (Match flow, Leaderboard, Match story).
Runs as the `football-live` job, triggered when new raw files arrive (at most every 5 minutes); `days_back` (default 1) limits it to recent files.

Live - Flattening

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import pandas as pd
import json, math
from datetime import datetime, timedelta, timezone

S3_BUCKET = "football-data-raw-01"
RAW_ROOT = f"s3://{S3_BUCKET}/live/raw"   # its own prefix: the football-live job is triggered by new files here

# Job parameter days_back: how many earlier UTC days of raw files to read besides today
# ("all" = every file, for a full rebuild). Every table below skips rows it already has
# (left anti join on kafka_partition + kafka_offset), so overlapping runs are safe.
dbutils.widgets.text("days_back", "1")
DAYS_BACK = dbutils.widgets.get("days_back").strip().lower()
if DAYS_BACK == "all":
    BRONZE_PATH = [f"{RAW_ROOT}/date=*/*.jsonl"]
else:
    _today = datetime.now(timezone.utc).date()
    _days = [(_today - timedelta(days=i)).isoformat() for i in range(int(DAYS_BACK) + 1)]
    _present = {f.name.rstrip("/") for f in dbutils.fs.ls(RAW_ROOT)}
    BRONZE_PATH = [f"{RAW_ROOT}/date={d}/*.jsonl" for d in _days if f"date={d}" in _present]
print("reading", BRONZE_PATH or "nothing")
DELTA_ROOT = f"s3://{S3_BUCKET}/live/live_delta"

RAW_SCHEMA = "kafka_partition LONG, kafka_offset LONG, kafka_timestamp_ms LONG, key STRING, value STRING"

TEAM_STATS = {
    "ballPossession": "possession", "expectedGoals": "xg", "bigChanceCreated": "big_chances",
    "totalShotsOnGoal": "shots_total", "shotsOffGoal": "shots_off_target", "goalkeeperSaves": "saves",
    "passes": "passes", "accuratePasses": "passes_accurate", "fouls": "fouls", "totalTackle": "tackles",
    "interceptionWon": "interceptions", "totalClearance": "clearances", "ballRecovery": "recoveries",
    "touchesInOppBox": "touches_opp_box", "finalThirdEntries": "final_third_entries", "duelWonPercent": "duels_won_pct",
}
PLAYER_STATS = {
    "minutesPlayed": "minutes", "touches": "touches", "totalPass": "passes", "accuratePass": "passes_accurate",
    "totalShots": "shots", "shotOffTarget": "shots_off_target", "expectedGoals": "xg", "expectedAssists": "xa",
    "duelWon": "duels_won", "duelLost": "duels_lost", "fouls": "fouls", "wasFouled": "was_fouled",
    "totalClearance": "clearances", "ballRecovery": "recoveries",
}

def _r3(v):
    return round(v, 3) if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v)) else v

def _jn(v):
    """jersey number -> float, or None (kept as a label, never used arithmetically)"""
    return float(v) if v is not None else None

def parse_channel(channel):
    """'match:16363651:stats' -> (16363651, 'stats'); non-match channels -> (None, None)"""
    parts = (channel or "").split(":")
    if len(parts) < 2 or parts[0] != "match" or not parts[1].isdigit():
        return None, None
    return int(parts[1]), (parts[2] if len(parts) > 2 else "match")

def flatten_team_stats(env, match_id, team_ids, offset_cols):
    p = env["data"]
    if "error" in p:                       
        return []
    ts = pd.to_datetime(env["timestamp"], unit="ms", utc=True)
    rows = []
    for block in p.get("statistics", []):
        if block["period"] != "ALL":
            continue
        for side in ("home", "away"):
            tid = team_ids.get(side)
            row = {**offset_cols, "match_id": match_id, "side": side,
                   "team_id": float(tid) if tid is not None else None,
                   "msg_type": env["type"], "event_ts": ts}
            row.update({col: None for col in TEAM_STATS.values()})
            for g in block["groups"]:
                for it in g["statisticsItems"]:
                    col = TEAM_STATS.get(it["key"])
                    if col and row[col] is None:     
                        v = it[f"{side}Value"]
                        row[col] = float(v) if v is not None else None
            rows.append(row)
    return rows

def flatten_lineups(env, match_id, team_ids, offset_cols):
    p = env["data"]
    ts = pd.to_datetime(env["timestamp"], unit="ms", utc=True)
    rows = []
    for side in ("home", "away"):
        team = p.get(side)
        if not team or not team.get("players"):        # error payload / not published yet
            continue
        ids = [x["teamId"] for x in team["players"] if x.get("teamId") is not None]
        side_team_id = max(set(ids), key=ids.count) if ids else None   # mode: an academy sub can carry a different teamId
        if side_team_id is not None:
            team_ids[side] = side_team_id                # feeds stats/incidents cells processed after this one

        for x in team["players"]:
            row = {
                **offset_cols, "match_id": match_id, "side": side,
                "team_id": float(side_team_id) if side_team_id is not None else None,
                "shirt_number": x.get("shirtNumber"), "player_name": x.get("player", {}).get("name"),
                "position": x.get("position"), "is_sub": x.get("substitute"), "is_captain": x.get("captain", False),
                "formation": team.get("formation"), "confirmed": p.get("confirmed"),
                "msg_type": env["type"], "event_ts": ts,
            }
            s = x.get("statistics", {})
            for k, col in PLAYER_STATS.items():
                v = s.get(k)
                v = float(v) if v is not None else None
                row[col] = _r3(v) if col in ("xg", "xa") else v
            rows.append(row)
    return rows

def flatten_incidents(env, match_id, team_ids, offset_cols):
    """One incidents message can hold a mix of types -> returns {table_name: [rows]}"""
    ts = pd.to_datetime(env["timestamp"], unit="ms", utc=True)
    out = {"live_goals": [], "live_cards": [], "live_subs": [], "live_var_decisions": [], "live_injury_time": []}

    for inc in env["data"].get("incidents", []):
        itype = inc.get("incidentType")
        base = {**offset_cols, "match_id": match_id, "time": inc.get("time"), "msg_type": env["type"], "event_ts": ts}

        if itype == "injuryTime":
            out["live_injury_time"].append({**base, "length": inc.get("length")})
            continue

        if itype not in ("goal", "card", "substitution", "varDecision") or inc.get("id") is None:
            continue                       

        side = "home" if inc.get("isHome") else "away"
        tid = team_ids.get(side)
        row = {**base, "id": inc["id"], "side": side, "team_id": float(tid) if tid is not None else None}

        if itype == "goal":
            row.update(home_score=inc.get("homeScore"), away_score=inc.get("awayScore"),
                       scorer_name=inc.get("player", {}).get("name"),
                       scorer_short_name=inc.get("player", {}).get("shortName"),
                       scorer_position=inc.get("player", {}).get("position"),
                       scorer_jersey_number=_jn(inc.get("player", {}).get("jerseyNumber")),
                       assist_name=inc.get("assist1", {}).get("name"),
                       assist_short_name=inc.get("assist1", {}).get("shortName"),
                       assist_position=inc.get("assist1", {}).get("position"),
                       assist_jersey_number=_jn(inc.get("assist1", {}).get("jerseyNumber")))
            out["live_goals"].append(row)
        elif itype == "card":
            row.update(incident_class=inc.get("incidentClass"), player_name=inc.get("playerName"),
                       reason=inc.get("reason"))
            out["live_cards"].append(row)
        elif itype == "substitution":
            row.update(player_in_name=inc.get("playerIn", {}).get("name"),
                       player_in_short_name=inc.get("playerIn", {}).get("shortName"),
                       player_in_position=inc.get("playerIn", {}).get("position"),
                       player_in_jersey_number=_jn(inc.get("playerIn", {}).get("jerseyNumber")),
                       player_out_name=inc.get("playerOut", {}).get("name"),
                       player_out_short_name=inc.get("playerOut", {}).get("shortName"),
                       player_out_position=inc.get("playerOut", {}).get("position"),
                       player_out_jersey_number=_jn(inc.get("playerOut", {}).get("jerseyNumber")))
            out["live_subs"].append(row)
        elif itype == "varDecision":
            row.update(incident_class=inc.get("incidentClass"), confirmed=inc.get("confirmed"))
            out["live_var_decisions"].append(row)

    return out

Live - Lineups

In [0]:
if not BRONZE_PATH:
    dbutils.notebook.exit("No raw live files in the selected days - nothing to do.")
df_bronze = spark.read.schema(RAW_SCHEMA).json(BRONZE_PATH)
pdf_bronze = df_bronze.toPandas()
team_ids_by_match = {}
rows = []
for _, rec in pdf_bronze.iterrows():
    env = json.loads(rec["value"])
    match_id, kind = parse_channel(env.get("channel"))
    if match_id is None or env.get("type") not in ("snapshot", "update") or kind != "lineups":
        continue
    team_ids = team_ids_by_match.setdefault(match_id, {})
    offset_cols = {"kafka_partition": int(rec["kafka_partition"]), "kafka_offset": int(rec["kafka_offset"])}
    rows += flatten_lineups(env, match_id, team_ids, offset_cols)

live_lineups_path = f"{DELTA_ROOT}/live_lineups"

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_lineups_path):
        existing = spark.read.format("delta").load(live_lineups_path).select("kafka_partition", "kafka_offset", "shirt_number")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "shirt_number"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_lineups_path)
    print(f"live_lineups: {candidate.count()} candidate rows -> {new_rows.count()} new")


Live - Stats

In [0]:

def load_team_ids():
    if not DeltaTable.isDeltaTable(spark, live_lineups_path):
        return {}
    df = spark.read.format("delta").load(live_lineups_path).select("match_id", "side", "team_id").distinct().toPandas()
    return {int(mid): dict(zip(g["side"], g["team_id"])) for mid, g in df.groupby("match_id")}

team_ids_by_match = load_team_ids()
rows = []
for _, rec in pdf_bronze.iterrows():
    env = json.loads(rec["value"])
    match_id, kind = parse_channel(env.get("channel"))
    if match_id is None or env.get("type") not in ("snapshot", "update") or kind != "stats":
        continue
    team_ids = team_ids_by_match.setdefault(match_id, {})
    offset_cols = {"kafka_partition": int(rec["kafka_partition"]), "kafka_offset": int(rec["kafka_offset"])}
    rows += flatten_team_stats(env, match_id, team_ids, offset_cols)

live_team_stats_path = f"{DELTA_ROOT}/live_team_stats"

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_team_stats_path):
        existing = spark.read.format("delta").load(live_team_stats_path).select("kafka_partition", "kafka_offset", "side")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "side"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_team_stats_path)
    print(f"live_team_stats: {candidate.count()} candidate rows -> {new_rows.count()} new")

Live - Incidents

In [0]:
team_ids_by_match = load_team_ids()
incident_rows = {"live_goals": [], "live_cards": [], "live_subs": [], "live_var_decisions": [], "live_injury_time": []}

for _, rec in pdf_bronze.iterrows():
    env = json.loads(rec["value"])
    match_id, kind = parse_channel(env.get("channel"))
    if match_id is None or env.get("type") not in ("snapshot", "update") or kind != "incidents":
        continue
    team_ids = team_ids_by_match.setdefault(match_id, {})
    offset_cols = {"kafka_partition": int(rec["kafka_partition"]), "kafka_offset": int(rec["kafka_offset"])}
    per_table = flatten_incidents(env, match_id, team_ids, offset_cols)
    for t, r in per_table.items():
        incident_rows[t] += r

print({t: len(r) for t, r in incident_rows.items()})

Live - Incidents - Goal

In [0]:
live_goals_path = f"{DELTA_ROOT}/live_goals"
rows = incident_rows["live_goals"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_goals_path):
        existing = spark.read.format("delta").load(live_goals_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_goals_path)
    print(f"live_goals: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_goals_path):
    display(spark.read.format("delta").load(live_goals_path))

Live - Incidents - Cards

In [0]:
live_cards_path = f"{DELTA_ROOT}/live_cards"
rows = incident_rows["live_cards"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_cards_path):
        existing = spark.read.format("delta").load(live_cards_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_cards_path)
    print(f"live_cards: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_cards_path):
    display(spark.read.format("delta").load(live_cards_path))

Live - Incidents - Var

In [0]:
live_var_decisions_path = f"{DELTA_ROOT}/live_var_decisions"
rows = incident_rows["live_var_decisions"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_var_decisions_path):
        existing = spark.read.format("delta").load(live_var_decisions_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_var_decisions_path)
    print(f"live_var_decisions: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_var_decisions_path):
    display(spark.read.format("delta").load(live_var_decisions_path))

Live - incidents - subs

In [0]:
live_subs_path = f"{DELTA_ROOT}/live_subs"
rows = incident_rows["live_subs"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_subs_path):
        existing = spark.read.format("delta").load(live_subs_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_subs_path)
    print(f"live_subs: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_subs_path):
    display(spark.read.format("delta").load(live_subs_path))

Live - Injury Time

In [0]:
live_injury_time_path = f"{DELTA_ROOT}/live_injury_time"
rows = incident_rows["live_injury_time"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_injury_time_path):
        existing = spark.read.format("delta").load(live_injury_time_path).select("kafka_partition", "kafka_offset", "time")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "time"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_injury_time_path)
    print(f"live_injury_time: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_injury_time_path):
    display(spark.read.format("delta").load(live_injury_time_path))